# 🏎️ JetBot 期末專案多功能控制面板 (整合主控最終版)

本筆記本將期末專案分為三個單元，便於實車落地時分步調試與最終合併展示。專案整合了 **Project 5 (道路跟隨)** 與 **Project 6 (路牌辨識)** 兩大自主駕駛模組。

### 📖 整合系統架構簡圖
```
CSI Camera (416x416) ➔ YOLOv4-tiny TRT ➔ 偵測路牌與寬度 ➔ 狀態機控制 (停止/減速/起步)
         │
         └───➔ OpenCV Resize (224x224) ➔ ResNet-18 TRT ➔ 預測 (X,Y) ➔ PD控制器 ➔ 馬達左右轉向
```

### 🚦 號誌對應動作參考表
| 號誌圖樣 | 類別名稱 | 說明 | 觸發動作 | 避雷防重複機制 |
| :---: | :---: | :--- | :--- | :--- |
| <img src="images/sign_detection/stop.png" width="40"> | **Class 0: stop** | 停車再開 | **停止 2 秒** 後重新起步 | 10秒時間冷卻鎖，防重複觸發卡死 |
| <img src="images/sign_detection/rail.png" width="40"> | **Class 1: rail** | 鐵路平交道 | **停止 5 秒** 後重新起步 | 10秒時間冷卻鎖，防重複觸發卡死 |
| <img src="images/sign_detection/pedestrian.png" width="40"> | **Class 2: pedestrian** | 當心行人 | **減速至 0.7x** 前進 | 通過後自動恢復巡航速度 |
| <img src="images/sign_detection/blocked.png" width="40"> | **Class 3: blocked** | 道路封閉 | **永久停止** 並關閉相機 | 煞停於路牌前方，不得越線 |

### 📊 模型訓練與驗證指標展示
<table>
  <tr>
    <td align="center"><b>道路跟隨預測效果 (綠點=標註, 紅點=預測)</b></td>
    <td align="center"><b>誤差分佈直方圖 (平均誤差 12.7 像素)</b></td>
  </tr>
  <tr>
    <td><img src="images/road_following/prediction_visual.png" width="380"></td>
    <td><img src="images/road_following/error_distribution.png" width="380"></td>
  </tr>
  <tr>
    <td align="center"><b>YOLOv4-tiny 收斂曲線</b></td>
    <td align="center"><b>測試集盲測預測結果 (mAP 達 97.06%)</b></td>
  </tr>
  <tr>
    <td><img src="images/sign_detection/results.png" width="380"></td>
    <td><img src="images/sign_detection/test_grid_all.jpg" width="380"></td>
  </tr>
</table>

---

## 🛠️ 使用說明與單元結構：
- **單元一：純道路跟隨測試 (測試 PID 與轉向)** — 僅載入 ResNet 模型，不佔用 YOLO 與 PyCUDA 資源，適合單獨微調 P/D Gain 滑桿。
- **單元二：純交通路牌辨識測試 (測試 YOLO 與動作)** — 僅載入 YOLO 模型，自走車直線前進，測試路標停/走動作是否正常。
- **單元三：合併最終版 (道路跟隨 + 路牌辨識)** — 雙模型並行推論，進行期末考完整跑道繞圈演示。

> ⚠️ **重要安全守則**：每當您要從一個單元切換到另一個單元時，**必須執行該單元最下方的「安全停止」單元格**，否則會出現相機被佔用 (Resource busy) 的錯誤！

---

In [3]:
## 🛠️ 零、道路跟隨 TensorRT 模型優化轉換 (選用)
# 0-1. 執行 TensorRT 編譯與轉換
import os
import subprocess
import torch
import torchvision.models as models

PYTORCH_MODEL = 'road_following_model/best_steering_model_xy.pth'
TRT_MODEL = 'road_following_model/best_steering_model_xy_trt.pth'
ONNX_MODEL = 'road_following_model/best_steering_model_xy.onnx'
ENGINE_MODEL = 'road_following_model/best_steering_model_xy.engine'

if not os.path.exists(PYTORCH_MODEL):
    print(f"❌ 錯誤：找不到 PyTorch 原始權重檔案：{PYTORCH_MODEL}")
    print("請確認您已將訓練好的 .pth 檔案上傳至對應目錄。")
else:
    print(f"正在載入 PyTorch 權重：{PYTORCH_MODEL}...")
    # 建立 ResNet-18 架構
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 2)
    model.load_state_dict(torch.load(PYTORCH_MODEL))
    
    # 搬移至 GPU 並設定為評估模式
    model = model.cuda().eval()
    
    # 1. 測試 PyTorch 運作是否正常
    print("正在進行 PyTorch 模型測試推論...")
    try:
        test_in = torch.zeros((1, 3, 224, 224)).cuda()
        test_out = model(test_in)
        print(f"✓ PyTorch 測試推論成功！輸出值：{test_out.cpu().detach().numpy()}")
    except Exception as e:
        print(f"❌ 錯誤：PyTorch 測試推論失敗！原因：{e}")
        
    # 2. 匯出為 ONNX 格式
    print("正在將 PyTorch 模型匯出為 ONNX 格式...")
    try:
        sample_data = torch.zeros((1, 3, 224, 224)).cuda()
        torch.onnx.export(
            model,
            sample_data,
            ONNX_MODEL,
            input_names=["input_0"],
            output_names=["output_0"],
            opset_version=11,
            do_constant_folding=True
        )
        print(f"✓ ONNX 匯出成功！已儲存至：{ONNX_MODEL}")
        has_onnx = True
    except Exception as e:
        print(f"❌ 錯誤：ONNX 匯出失敗：{e}")
        has_onnx = False

    # 3. 使用 Nvidia 官方 trtexec 編譯器將 ONNX 編譯為 TensorRT .engine
    if has_onnx and os.path.exists(ONNX_MODEL):
        print("\n正在將 ONNX 編譯為 TensorRT 引擎 (這在 JetBot 上需要 1~2 分鐘，請耐心等候)...\n")
        print("提示：trtexec 會在獨立行程中執行，大幅降低 VRAM 記憶體不足 (OOM) 的機率。")
        
        # 尋找 trtexec 路徑
        trtexec_path = "/usr/src/tensorrt/bin/trtexec"
        if not os.path.exists(trtexec_path):
            trtexec_path = "trtexec"
            
        # 嘗試 FP16 半精度模式編譯
        print("嘗試：FP16 半精度模式編譯...")
        cmd_fp16 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL} --fp16"
        print(f"執行：{cmd_fp16}")
        
        res = subprocess.run(cmd_fp16, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
        if res.returncode != 0:
            print("⚠️ 警告：FP16 編譯失敗，正在嘗試自動降級至 FP32 單精度模式編譯...")
            cmd_fp32 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL}"
            print(f"執行：{cmd_fp32}")
            res = subprocess.run(cmd_fp32, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            
        if res.returncode == 0:
            print("✓ TensorRT 引擎編譯成功！")
            
            # 4. 讀取編譯好的引擎位元組，包裝成 torch2trt 專用的 TRTModule state_dict 格式
            print(f"正在將 .engine 包裝為 PyTorch 相容的 {TRT_MODEL}...")
            try:
                with open(ENGINE_MODEL, 'rb') as f:
                    engine_bytes = f.read()
                
                state_dict = {
                    "engine": bytearray(engine_bytes),
                    "input_names": ["input_0"],
                    "output_names": ["output_0"]
                }
                
                os.makedirs(os.path.dirname(TRT_MODEL), exist_ok=True)
                torch.save(state_dict, TRT_MODEL)
                print(f"✓ 轉換與儲存成功！TensorRT 加速模型已就緒。")
            except Exception as e:
                print(f"❌ 錯誤：包裝與儲存 TRTModule 失敗：{e}")
        else:
            print("❌ 錯誤：TensorRT 引擎編譯失敗！詳細資訊：")
            print(res.stderr.decode('utf-8', errors='ignore'))
            print("\n請嘗試重啟 Jupyter Kernel 以釋放更多系統資源。")


# 0-2. Build YOLO TensorRT engine on JetBot
YOLO_DIR = 'yolo'
YOLO_MODEL_NAME = 'yolov4-tiny-416'
YOLO_ENGINE_CANDIDATES = [
    os.path.join(YOLO_DIR, YOLO_MODEL_NAME + '.trt'),
    os.path.join(YOLO_DIR, YOLO_MODEL_NAME + '.engine')
]

if any(os.path.exists(path) for path in YOLO_ENGINE_CANDIDATES):
    print('YOLO TensorRT engine already exists. Conversion skipped.')
else:
    yolo_to_onnx = os.path.join(YOLO_DIR, 'yolo_to_onnx.py')
    onnx_to_tensorrt = os.path.join(YOLO_DIR, 'onnx_to_tensorrt.py')
    required_yolo_assets = [
        os.path.join(YOLO_DIR, YOLO_MODEL_NAME + '.cfg'),
        os.path.join(YOLO_DIR, YOLO_MODEL_NAME + '.weights'),
        os.path.join(YOLO_DIR, 'obj.names')
    ]
    missing_assets = [path for path in required_yolo_assets if not os.path.exists(path)]
    missing_tools = [path for path in [yolo_to_onnx, onnx_to_tensorrt] if not os.path.exists(path)]

    if missing_assets:
        print('Missing YOLO deployment assets:', missing_assets)
    elif missing_tools:
        print('Missing JetBot conversion tools:', missing_tools)
        print('Copy this package into the trt_yolov4-tiny-master root directory.')
    else:
        python_executable = 'python3'
        print('Converting YOLO weights to ONNX...')
        yolo_onnx_result = subprocess.run(
            [python_executable, 'yolo_to_onnx.py', '-c', '4', '-m', YOLO_MODEL_NAME],
            cwd=YOLO_DIR
        )
        if yolo_onnx_result.returncode != 0:
            print('YOLO ONNX conversion failed.')
        else:
            print('Building YOLO TensorRT engine...')
            yolo_trt_result = subprocess.run(
                [python_executable, 'onnx_to_tensorrt.py', '-c', '4', '-m', YOLO_MODEL_NAME],
                cwd=YOLO_DIR
            )
            if yolo_trt_result.returncode == 0:
                print('YOLO TensorRT conversion completed.')
            else:
                print('YOLO TensorRT conversion failed.')

Missing YOLO deployment assets: ['yolo/obj.names']


---

## 🏁 單元一：純道路跟隨測試 (調校 PID 參數)
此部分只載入 Project 5 的道路循跡模型，適合微調馬達轉向與 PD 參數。

In [6]:
# A1. 匯入基礎套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

# A2. 載入 ResNet-18 TensorRT 模型
device = torch.device('cuda')
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
print("單元一：道路跟隨模型載入成功 ✓")

單元一：道路跟隨模型載入成功 ✓


In [7]:
# A3. 初始化相機 (224x224) 與 Robot
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    try:
        jetbot.Camera.instance().stop()
    except:
        pass
    del jetbot.Camera._instance
camera_r = Camera.instance(width=224, height=224, fps=10)
robot_r = Robot()
print("相機 (224x224) 與馬達初始化完成 ✓")

RuntimeError: Could not initialize camera.  Please see error trace.

In [6]:
# A4. 定義影像處理與滑桿介面
mean_r = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_r = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_only(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_r[:, None, None]) / std_r[:, None, None]
    return image.unsqueeze(0)

# 建立控制滑桿
speed_slider_r = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.02, description='D Gain')
bias_slider_r = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')

# 建立遙測數據顯示
x_text_r = widgets.Label(value="X: 0.000")
y_text_r = widgets.Label(value="Y: 0.000")
steer_text_r = widgets.Label(value="Steering: 0.000")
motor_text_r = widgets.Label(value="L: 0.00, R: 0.00")

# 即時影像 Widget
image_widget_r = widgets.Image(format='jpeg', width=300, height=300)

# 左右排版與顯示
controls_box_r = widgets.VBox([
    widgets.HTML("<b>⚙️ PID 參數調校控制台</b>"),
    speed_slider_r,
    p_slider_r,
    d_slider_r,
    bias_slider_r,
    widgets.HTML("<hr><b>📊 即時遙測數據</b>"),
    x_text_r,
    y_text_r,
    steer_text_r,
    motor_text_r
])

panel_r = widgets.HBox([image_widget_r, controls_box_r])
display(panel_r)

In [7]:
# A5. 循跡控制迴圈
angle_last_r = 0.0

def execute_road_only(change):
    global angle_last_r
    image = change['new']
    
    # 影像前處理與推論
    road_tensor = preprocess_road_only(image)
    output = model_road(road_tensor).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[1])
    
    # 計算前向距離與目標角度 (避免除以零或方向反轉)
    # y_forward = 1.0 - y。當 y 越小(在頂部/遠處)，y_forward 越大(前向距離遠)
    y_forward = max(1.0 - y, 0.05)
    angle = np.arctan2(x, y_forward)
    
    # PD 控制器計算轉向
    pid = angle * p_slider_r.value + (angle - angle_last_r) * d_slider_r.value
    angle_last_r = angle
    steering = pid + bias_slider_r.value
    
    # 計算馬達出力
    base_speed = speed_slider_r.value
    left_motor = max(min(base_speed + steering, 1.0), 0.0)
    right_motor = max(min(base_speed - steering, 1.0), 0.0)
    
    # 驅動馬達
    robot_r.left_motor.value = left_motor
    robot_r.right_motor.value = right_motor
    
    # 更新即時數據顯示
    x_text_r.value = f"X: {x:.3f}"
    y_text_r.value = f"Y: {y:.3f}"
    steer_text_r.value = f"Steering: {steering:.3f}"
    motor_text_r.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
    
    # 繪製綠色路徑點
    display_img = np.copy(image)
    pixel_x = int(x * 112 + 112)
    pixel_y = int(y * 112 + 112)
    cv2.circle(display_img, (pixel_x, pixel_y), 8, (0, 255, 0), -1)
    image_widget_r.value = bgr8_to_jpeg(display_img)

print("單元一控制函式 execute_road_only 定義完成 ✓")

單元一控制函式 execute_road_only 定義完成 ✓


In [8]:
# A6. 啟動單元一 (道路跟隨)
execute_road_only({'new': camera_r.value})
camera_r.observe(execute_road_only, names='value')
print("🚗 單元一：純道路跟隨自動駕駛啟動！")

🚗 單元一：純道路跟隨自動駕駛啟動！


In [9]:
# A7. 安全停止單元一 (切換單元前必執行！)
camera_r.unobserve(execute_road_only, names='value')
time.sleep(0.5)
robot_r.stop()
camera_r.stop()
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    del jetbot.Camera._instance
print("單元一已停止，相機資源已釋放 ✓")

單元一已停止，相機資源已釋放 ✓


---

## 🚥 單元二：純交通路牌辨識測試
此部分只載入 Project 6 的 YOLO 號誌偵測模型，測試自走車看到路牌後的停、走動作是否完全正確。在此模式下，自走車會直線行駛，只反應路牌動作。

In [ ]:
# B1. 匯入 YOLO 相關套件
import os
import time
import cv2
import numpy as np
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import pycuda.autoinit
from utils.yolo import TRT_YOLO

# B2. 載入 YOLO 模型
model_sign_only = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("單元二：路牌辨識模型載入成功 ✓")

In [ ]:
# B3. 初始化相機 (416x416) 與 Robot
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    try:
        jetbot.Camera.instance().stop()
    except:
        pass
    del jetbot.Camera._instance
camera_s = Camera.instance(width=416, height=416, fps=10)
robot_s = Robot()
print("相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# B4. Sign detection controls
speed_slider_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Base Speed')
stop_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Stop Width')
rail_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=30, description='Rail Width')
ped_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Ped Width')
blocked_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Block Width')
image_widget_s = widgets.Image(format='jpeg', width=416, height=416)

sign_threshold_box_s = widgets.VBox([
    widgets.HTML("<b>Per-class trigger width</b>"),
    stop_width_slider_s,
    rail_width_slider_s,
    ped_width_slider_s,
    blocked_width_slider_s
])
display(widgets.HBox([image_widget_s, widgets.VBox([speed_slider_s, sign_threshold_box_s])]))

In [ ]:
# B5. Sign detection control loop
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False
sign_confirm_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}
SIGN_CONFIRM_FRAMES_S = {0: 2, 1: 2, 2: 2, 3: 2}
SIGN_PRIORITY_S = {3: 4, 0: 3, 1: 2, 2: 1}

def _execute_sign_only_core(change):
    global stop_cooldown_s, rail_cooldown_s, pedestrian_until_s, blocked_latched_s, safety_latched_s
    img = change['new']
    current_time = time.monotonic()

    # Keep the robot stopped after BLOCKED until the start cell resets the latch.
    if blocked_latched_s or safety_latched_s:
        robot_s.stop()
        return

    boxes, confs, clss = model_sign_only.detect(img, conf_th=0.6)
    detected_signs = []
    for box, conf, cls in zip(boxes, confs, clss):
        cid = int(cls)
        width = int(box[2] - box[0])
        detected_signs.append({
            "width": width,
            "class_id": cid,
            "confidence": float(conf),
            "box": box
        })
    detected_signs.sort(reverse=True, key=lambda x: x["width"])

    alert_widths = {
        0: stop_width_slider_s.value,
        1: rail_width_slider_s.value,
        2: ped_width_slider_s.value,
        3: blocked_width_slider_s.value
    }

    # Keep only the nearest qualifying detection for each class.
    qualifying = {}
    for sign in detected_signs:
        cid = sign["class_id"]
        if cid not in alert_widths:
            continue
        if cid == 0 and current_time < stop_cooldown_s:
            continue
        if cid == 1 and current_time < rail_cooldown_s:
            continue
        if sign["width"] >= alert_widths[cid] and cid not in qualifying:
            qualifying[cid] = sign

    # Require consecutive frames to reduce false or transient triggers.
    for cid in sign_confirm_counts_s:
        if cid in qualifying:
            sign_confirm_counts_s[cid] = min(
                sign_confirm_counts_s[cid] + 1,
                SIGN_CONFIRM_FRAMES_S[cid]
            )
        else:
            sign_confirm_counts_s[cid] = 0

    confirmed = [
        sign for cid, sign in qualifying.items()
        if sign_confirm_counts_s[cid] >= SIGN_CONFIRM_FRAMES_S[cid]
    ]
    confirmed.sort(
        reverse=True,
        key=lambda x: (SIGN_PRIORITY_S[x["class_id"]], x["width"])
    )
    active_sign = confirmed[0] if confirmed else None

    current_action = "Driving Straight"
    if active_sign is not None:
        cid = active_sign["class_id"]
        sign_confirm_counts_s[cid] = 0

        if cid == 0:
            print("[SIGN] Confirmed STOP. Stopping for 2s...")
            robot_s.stop()
            time.sleep(2.0)
            stop_cooldown_s = time.monotonic() + 10.0
            return  # Wait for a fresh frame after the stop interval.
        elif cid == 1:
            print("[SIGN] Confirmed RAILWAY. Stopping for 5s...")
            robot_s.stop()
            time.sleep(5.0)
            rail_cooldown_s = time.monotonic() + 10.0
            return  # Wait for a fresh frame after the stop interval.
        elif cid == 2:
            pedestrian_until_s = time.monotonic() + 1.0
        elif cid == 3:
            print("[SIGN] Confirmed BLOCKED. Permanent stop latched.")
            blocked_latched_s = True
            robot_s.stop()
            return

    speed_multiplier = 0.7 if current_time < pedestrian_until_s else 1.0
    if speed_multiplier < 1.0:
        current_action = "ACTION: Pedestrian (Slow 0.7x)"

    base_speed = speed_slider_s.value * speed_multiplier
    robot_s.left_motor.value = base_speed
    robot_s.right_motor.value = base_speed

    display_img = np.copy(img)
    for sign in detected_signs:
        box = sign["box"]
        cid = sign["class_id"]
        label = f"ID:{cid} C:{sign['confidence']:.2f} W:{sign['width']}"
        cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
        cv2.putText(display_img, label, (box[0], max(box[1] - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
    cv2.putText(display_img, current_action, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    image_widget_s.value = bgr8_to_jpeg(display_img)

def execute_sign_only(change):
    global safety_latched_s
    try:
        _execute_sign_only_core(change)
    except Exception as exc:
        safety_latched_s = True
        robot_s.stop()
        print(f"[SAFETY] Unit 2 callback failed; motors stopped: {exc}")

print("Unit 2 execute_sign_only is ready.")

In [ ]:
# B6. Start Unit 2 (sign detection)
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False
sign_confirm_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}
execute_sign_only({'new': camera_s.value})
camera_s.observe(execute_sign_only, names='value')
print("Unit 2 sign detection started.")

In [ ]:
# B7. 安全停止單元二 (切換單元前必執行！)
camera_s.unobserve(execute_sign_only, names='value')
time.sleep(0.5)
robot_s.stop()
camera_s.stop()
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    del jetbot.Camera._instance
print("單元二已停止，相機資源已釋放 ✓")

---

In [ ]:
# C1. 匯入完整套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import pycuda.autoinit
from utils.yolo import TRT_YOLO

# C2. 同時載入兩個 TensorRT 模型
device = torch.device('cuda')
model_road_final = TRTModule()
model_road_final.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
print("道路模型載入完成 ✓")

model_sign_final = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("路牌模型載入完成 ✓")

In [ ]:
# C3. 初始化相機 (416x416) 與 Robot
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    try:
        jetbot.Camera.instance().stop()
    except:
        pass
    del jetbot.Camera._instance
camera_f = Camera.instance(width=416, height=416, fps=10)
robot_f = Robot()
print("最終合併版相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# C4. Preprocessing and control panel
mean_f = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_f = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_final(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_f[:, None, None]) / std_f[:, None, None]
    return image.unsqueeze(0)

# Control sliders
speed_slider_f = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.02, description='D Gain')
bias_slider_f = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')
stop_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Stop Width')
rail_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=30, description='Rail Width')
ped_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Ped Width')
blocked_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Block Width')

# Telemetry labels
x_text_f = widgets.Label(value="X: 0.000")
y_text_f = widgets.Label(value="Y: 0.000")
steer_text_f = widgets.Label(value="Steering: 0.000")
motor_text_f = widgets.Label(value="L: 0.00, R: 0.00")
action_text_f = widgets.Label(value="Action: Following Lane")

image_widget_f = widgets.Image(format='jpeg', width=416, height=416)

controls_box_f = widgets.VBox([
    widgets.HTML("<b>Combined autonomous driving controls</b>"),
    speed_slider_f,
    p_slider_f,
    d_slider_f,
    bias_slider_f,
    widgets.HTML("<hr><b>Per-class sign trigger width</b>"),
    stop_width_slider_f,
    rail_width_slider_f,
    ped_width_slider_f,
    blocked_width_slider_f,
    widgets.HTML("<hr><b>Live telemetry</b>"),
    x_text_f,
    y_text_f,
    steer_text_f,
    motor_text_f,
    action_text_f
])

panel_f = widgets.HBox([image_widget_f, controls_box_f])
display(panel_f)

In [ ]:
# C5. Combined autonomous driving loop
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False
sign_confirm_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}
SIGN_CONFIRM_FRAMES_F = {0: 2, 1: 2, 2: 2, 3: 2}
SIGN_PRIORITY_F = {3: 4, 0: 3, 1: 2, 2: 1}

def _execute_final_core(change):
    global angle_last_f, stop_cooldown_f, rail_cooldown_f
    global pedestrian_until_f, blocked_latched_f, safety_latched_f

    img = change['new']
    current_time = time.monotonic()

    # Keep the robot stopped after BLOCKED until the start cell resets the latch.
    if blocked_latched_f or safety_latched_f:
        robot_f.stop()
        action_text_f.value = "Action: BLOCKED LATCHED"
        return

    # 1. Detect traffic signs.
    boxes, confs, clss = model_sign_final.detect(img, conf_th=0.6)
    detected_signs = []
    for box, conf, cls in zip(boxes, confs, clss):
        cid = int(cls)
        width = int(box[2] - box[0])
        detected_signs.append({
            "width": width,
            "class_id": cid,
            "confidence": float(conf),
            "box": box
        })
    detected_signs.sort(reverse=True, key=lambda x: x["width"])

    alert_widths = {
        0: stop_width_slider_f.value,
        1: rail_width_slider_f.value,
        2: ped_width_slider_f.value,
        3: blocked_width_slider_f.value
    }

    qualifying = {}
    for sign in detected_signs:
        cid = sign["class_id"]
        if cid not in alert_widths:
            continue
        if cid == 0 and current_time < stop_cooldown_f:
            continue
        if cid == 1 and current_time < rail_cooldown_f:
            continue
        if sign["width"] >= alert_widths[cid] and cid not in qualifying:
            qualifying[cid] = sign

    # Require two consecutive frames before triggering an action.
    for cid in sign_confirm_counts_f:
        if cid in qualifying:
            sign_confirm_counts_f[cid] = min(
                sign_confirm_counts_f[cid] + 1,
                SIGN_CONFIRM_FRAMES_F[cid]
            )
        else:
            sign_confirm_counts_f[cid] = 0

    confirmed = [
        sign for cid, sign in qualifying.items()
        if sign_confirm_counts_f[cid] >= SIGN_CONFIRM_FRAMES_F[cid]
    ]
    confirmed.sort(
        reverse=True,
        key=lambda x: (SIGN_PRIORITY_F[x["class_id"]], x["width"])
    )
    active_sign = confirmed[0] if confirmed else None

    if active_sign is not None:
        cid = active_sign["class_id"]
        sign_confirm_counts_f[cid] = 0

        if cid == 0:
            action_text_f.value = "Action: Stop (2s)"
            print("[SIGN] Confirmed STOP. Stopping for 2s...")
            robot_f.stop()
            time.sleep(2.0)
            stop_cooldown_f = time.monotonic() + 10.0
            return  # Wait for a fresh frame after the stop interval.
        elif cid == 1:
            action_text_f.value = "Action: Railway (5s)"
            print("[SIGN] Confirmed RAILWAY. Stopping for 5s...")
            robot_f.stop()
            time.sleep(5.0)
            rail_cooldown_f = time.monotonic() + 10.0
            return  # Wait for a fresh frame after the stop interval.
        elif cid == 2:
            pedestrian_until_f = time.monotonic() + 1.0
        elif cid == 3:
            action_text_f.value = "Action: BLOCKED LATCHED"
            print("[SIGN] Confirmed BLOCKED. Permanent stop latched.")
            blocked_latched_f = True
            robot_f.stop()
            return

    speed_multiplier = 0.7 if current_time < pedestrian_until_f else 1.0
    current_action = "Pedestrian (Slow 0.7x)" if speed_multiplier < 1.0 else "Following Lane"

    # 2. Run road-following inference.
    road_tensor = preprocess_road_final(img)
    output = model_road_final(road_tensor).detach().cpu().numpy().flatten()
    if len(output) < 2 or not np.all(np.isfinite(output[:2])):
        raise ValueError(f"Invalid road model output: {output}")
    x = float(np.clip(output[0], -1.0, 1.0))
    y = float(np.clip(output[1], -1.0, 1.0))

    y_forward = max(1.0 - y, 0.05)
    angle = np.arctan2(x, y_forward)

    pid = angle * p_slider_f.value + (angle - angle_last_f) * d_slider_f.value
    angle_last_f = angle
    steering = pid + bias_slider_f.value

    # Reduce speed in sharp turns; retain at least 60% of the configured speed.
    curve_scale = max(0.60, 1.0 - 1.5 * abs(steering))
    base_speed = speed_slider_f.value * speed_multiplier * curve_scale
    left_motor = max(min(base_speed + steering, 1.0), 0.0)
    right_motor = max(min(base_speed - steering, 1.0), 0.0)

    robot_f.left_motor.value = left_motor
    robot_f.right_motor.value = right_motor

    x_text_f.value = f"X: {x:.3f}"
    y_text_f.value = f"Y: {y:.3f}"
    steer_text_f.value = f"Steering: {steering:.3f} | Curve speed: {curve_scale:.2f}x"
    motor_text_f.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
    action_text_f.value = f"Action: {current_action}"

    # 3. Draw telemetry overlays.
    display_img = np.copy(img)
    pixel_x = int(x * 208 + 208)
    pixel_y = int(y * 208 + 208)
    cv2.circle(display_img, (pixel_x, pixel_y), 10, (0, 255, 0), -1)

    for sign in detected_signs:
        box = sign["box"]
        cid = sign["class_id"]
        label = f"ID:{cid} C:{sign['confidence']:.2f} W:{sign['width']}"
        cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
        cv2.putText(display_img, label, (box[0], max(box[1] - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)

    cv2.putText(display_img, current_action, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    image_widget_f.value = bgr8_to_jpeg(display_img)

def execute_final(change):
    global safety_latched_f
    try:
        _execute_final_core(change)
    except Exception as exc:
        safety_latched_f = True
        robot_f.stop()
        action_text_f.value = "Action: SAFETY STOP"
        print(f"[SAFETY] Unit 3 callback failed; motors stopped: {exc}")

print("Unit 3 execute_final is ready.")

In [ ]:
# C6. Start Unit 3 (combined autonomous driving)
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False
sign_confirm_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}
execute_final({'new': camera_f.value})
camera_f.observe(execute_final, names='value')
print("Unit 3 combined autonomous driving started.")

In [ ]:
# C7. 安全停止單元三
camera_f.unobserve(execute_final, names='value')
time.sleep(0.5)
robot_f.stop()
camera_f.stop()
import jetbot
if hasattr(jetbot.Camera, '_instance'):
    del jetbot.Camera._instance
print("單元三已停止，相機與馬達資源已安全釋放 ✓")